# Fresh HotpotQA SDPO training in Colab

Before running, choose **Runtime → Disconnect and delete runtime**, reconnect, and select an **L4 GPU**. The HotpotQA JSONL files and source configuration are expected to already exist in Google Drive. Run every cell in order.

In [ ]:
import os
import sys
from urllib.parse import urlunparse

from google.colab import drive, userdata

drive.mount("/content/drive")

hf_token = userdata.get("HF_TOKEN")
hf_infer = userdata.get("HF_INFER")
assert hf_token, "HF_TOKEN is missing from Colab Secrets"
assert hf_infer, "HF_INFER is missing from Colab Secrets"

os.environ["HF_TOKEN"] = hf_token
os.environ["OPENAI_API_KEY"] = hf_infer
os.environ["OPENAI_BASE_URL"] = urlunparse(
    ("https", "router.huggingface.co", "/v1", "", "", "")
)
os.environ["PYTHONUNBUFFERED"] = "1"

assert os.environ["OPENAI_BASE_URL"] == "https://router.huggingface.co/v1"
assert "[" not in os.environ["OPENAI_BASE_URL"]
assert "]" not in os.environ["OPENAI_BASE_URL"]

import torch

assert torch.cuda.is_available(), "No GPU detected"
assert torch.cuda.is_bf16_supported(), "Select a bf16-capable GPU such as L4 or A100"

print("Python:", sys.executable)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("Actual base URL:", repr(os.environ["OPENAI_BASE_URL"]))

## Clone the pinned repository and install dependencies

In [ ]:
%cd /content
!git clone https://github.com/mmzinn12/rlm-ib.git
%cd /content/rlm-ib
!git checkout 18cf8b7d7b78a088eb1383f005f1e1cd6f4f165e
!git rev-parse HEAD

In [ ]:
%pip install -q -e /content/rlm-ib -e "/content/rlm-ib/training[colab,hub-datasets]"

In [ ]:
import importlib.metadata as metadata
import openai
import rlm
import rlm_train

print("OpenAI:", metadata.version("openai"))
print("RLM:", metadata.version("rlms"))
print("RLM Train:", metadata.version("rlm-train"))

## Verify package sanity checks

Run the SDPO package tests (including teacher-target normalization) and print the results.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_sdpo.py", "-q"],
    cwd="/content/rlm-ib/training",
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Package SDPO sanity checks failed"
print("Package SDPO sanity checks passed.")


## Configure the full pipeline

The run is configured with two dicts — `run_config` (validated into the canonical `RunSpec`) and `experiment_config` (run-only knobs in `ExperimentSettings`). Each has a descriptive section and its own cell below; edit the values there. Both are written to JSON on Drive and read back, so the notebook and the `rlm_train` CLI share one config.

**Path pointers** (plain variables, set in the next cell)

| Variable | Significance | Example |
| --- | --- | --- |
| `TRAIN_PATH` | JSONL file the student trains on. | `/content/drive/MyDrive/rlm-ib-data/hotpotqa-dense-train.jsonl` |
| `EVAL_PATH` | Held-out JSONL scored during evaluation. | `/content/drive/MyDrive/rlm-ib-data/hotpotqa-dense-eval.jsonl` |
| `OUTPUT_ROOT` | Parent folder for run outputs (metrics, rollouts, predictions). | `/content/drive/MyDrive/rlm-ib-outputs` |
| `RUN_NAME` | Unique run name; names the output subfolder and config files. | `hotpotqa-sdpo-qwen25math15b-bf16-fresh` |

Each JSONL line is one task: `{"id": "q1", "prompt": "<dense context + question>", "target": "<gold answer>"}` (optional `"metadata": {}`). `prompt` becomes the public task; `target` is kept as verifier-only data.

In [ ]:
import os
from pathlib import Path

from rlm_train.experiment import ExperimentSettings
from rlm_train.spec.run import RunSpec

CONFIG_DIR = Path("/content/drive/MyDrive/rlm-ib-configs")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# --- Edit these pointers for your run ---
TRAIN_PATH = "/content/drive/MyDrive/rlm-ib-data/hotpotqa-dense-train.jsonl"
EVAL_PATH = "/content/drive/MyDrive/rlm-ib-data/hotpotqa-dense-eval.jsonl"
OUTPUT_ROOT = "/content/drive/MyDrive/rlm-ib-outputs"
RUN_NAME = "hotpotqa-sdpo-qwen25math15b-bf16-fresh"

assert Path(TRAIN_PATH).exists(), f"Missing training data: {TRAIN_PATH}"
assert Path(EVAL_PATH).exists(), f"Missing evaluation data: {EVAL_PATH}"


### `run_config` — the canonical `RunSpec`

Hyperparameters, dataset pointing, judge, objectives, and evaluation. Validated into a frozen `RunSpec`. Edit values in the cell below.


#### Student


**`student`** — the trainable policy (the model that generates and is optimized)

| Key | Significance | Example |
| --- | --- | --- |
| `model_id` | Hugging Face model to load as the student. | `Qwen/Qwen2.5-Math-1.5B` |
| `policy_owner` | Identity tag marking which tokens are the student's (trainable). | `student` |


In [ ]:
student_config = {
        "model_id": "Qwen/Qwen2.5-Math-1.5B", 
        "policy_owner": "student"
        }

#### Rollout


**`rollout`** — how the recursive policy executes per task

| Key | Significance | Example |
| --- | --- | --- |
| `environment` | REPL/execution environment for the RLM. | `local` |
| `max_depth` | Deepest level of recursive sub-questions allowed. | `2` |
| `max_iterations` | Max reasoning/subcall steps per node before stopping. | `6` |


In [ ]:
rollout_config = {
                "environment": "local", 
                "max_depth": 2, 
                "max_iterations": 6
                }

#### Judge


**`judge`** — scores helper-question quality to produce feedback

| Key | Significance | Example |
| --- | --- | --- |
| `provider` | `openai` for a real LLM judge, or `fake` for a deterministic stub. | `openai` |
| `model` | Judge model route (for `openai`). | `Qwen/Qwen2.5-7B-Instruct` |
| `mode` | `categorical` (enum labels, reliable) or `full` (bounded numeric scores). | `categorical` |
| `api_key_environment` | Env var holding the judge API key. | `OPENAI_API_KEY` |
| `base_url` | OpenAI-compatible endpoint (HF router here). | `os.environ["OPENAI_BASE_URL"]` |


In [ ]:
judge_config = {
        "provider": "openai",
        "model": "Qwen/Qwen2.5-7B-Instruct",
        "mode": "categorical",  # or "full"
        "api_key_environment": "OPENAI_API_KEY",
        "base_url": os.environ["OPENAI_BASE_URL"],
    }

#### Teacher


**`teacher`** — source of the distillation targets

| Key | Significance | Example |
| --- | --- | --- |
| `strategy` | `current_policy` (self-distillation) is wired; `ema`/`fixed`/`external` are not yet. | `current_policy` |


In [ ]:
teacher_config = {
                "strategy": "current_policy"
                }

#### SDPO Objective


**`objectives.sdpo`** — the SDPO training objective

| Key | Significance | Example |
| --- | --- | --- |
| `enabled` | Turn the SDPO objective on. | `True` |
| `weight` | Loss weight when composing objectives (must be > 0 when enabled). | `1.0` |
| `token_scope` | Which student tokens get gradient: `helper_questions`, `natural_language`, `subcall_natural_language`, or `all_student_tokens`. | `helper_questions` |
| `feedback_scope` | Evidence the judge may see: `causal_local`, `retrospective_local`, or `privileged_diagnostic`. | `retrospective_local` |
| `top_k` | Size of the teacher's top-k support kept per token (plus a tail bucket). | `100` |


In [ ]:
sdpo_config = {
            "enabled": True,
            "weight": 1.0,
            "token_scope": "helper_questions",
            "feedback_scope": "retrospective_local",
            "top_k": 100,
        }


In [ ]:

objectives_config = {
            "sdpo": sdpo_config
            }

#### Datasets


**`training_dataset` / `evaluation_datasets`** — data pointers

| Key | Significance | Example |
| --- | --- | --- |
| `adapter` | Loader for the file (only `jsonl` is wired). | `jsonl` |
| `source` | Path to the JSONL file. | `TRAIN_PATH` / `EVAL_PATH` |
| `split` | Split label recorded for provenance. | `train` / `test` |


In [ ]:
training_dataset_config = {
                "adapter": "jsonl", 
                "source": TRAIN_PATH, 
                "split": "train"
                }

In [ ]:
jsonl_config = {"adapter": "jsonl",
                "source": EVAL_PATH,
                "split": "test"
                }
evaluation_dataset_config = [
                            jsonl_config
                            ]

#### Evaluation


**`evaluation`** — held-out scoring settings

| Key | Significance | Example |
| --- | --- | --- |
| `base_seed` | Base sampling seed for evaluation rollouts. | `0` |
| `samples_per_problem` | Rollouts generated per held-out question. | `1` |


In [ ]:
eval_config = {
                    "base_seed": 0, 
                    "samples_per_problem": 1
                    }

#### Artifacts


**`artifacts`**

| Key | Significance | Example |
| --- | --- | --- |
| `output_directory` | Where metrics, rollouts, and predictions are written. | `f"{OUTPUT_ROOT}/{RUN_NAME}"` |


In [ ]:
artifacts_config = {
                "output_directory": f"{OUTPUT_ROOT}/{RUN_NAME}"
                }

#### Runtime


**`runtime`** — optimization and hardware

| Key | Significance | Example |
| --- | --- | --- |
| `precision` | Model dtype: `bf16`, `fp16`, or `fp32`. | `bf16` |
| `learning_rate` | Peak AdamW learning rate. | `5e-5` |
| `max_gradient_norm` | Gradient-norm clip threshold. | `1.0` |
| `gradient_accumulation_steps` | Micro-batches accumulated before each optimizer step. | `2` |
| `max_optimizer_steps` | Total optimizer steps to run. | `25` |
| `seed` | Base RNG seed. | `0` |
| `warmup_steps` *(optional)* | LR warmup steps; `0` disables scheduling (constant LR). | `5` |
| `scheduler` *(optional)* | LR schedule shape when warmup > 0: `constant`, `linear`, `cosine`. | `linear` |

In [ ]:
runtime_config = {
        "precision": "bf16",
        "learning_rate": 5e-5,
        "max_gradient_norm": 1.0,
        "gradient_accumulation_steps": 2,
        "max_optimizer_steps": 25,
        "seed": 0,
    }

#### Full Config

In [ ]:
# Hyperparameters, dataset pointing, judge, objectives, and evaluation in one place.
run_config = {
    "student": student_config,
    "rollout": rollout_config,
    "judge": judge_config,
    "teacher": teacher_config,
    "objectives": 
        {
        "sdpo": sdpo_config
        },
    "training_dataset": training_dataset_config,
    "evaluation_datasets": evaluation_dataset_config,
    "evaluation": eval_config,
    "artifacts": artifacts_config,
    "runtime": runtime_config,
}



: 

### `experiment_config` — run-only knobs (`ExperimentSettings`)

Readout and scoring settings that are outside the canonical `RunSpec`. Edit values in the cell below.

| Key | Significance | Example |
| --- | --- | --- |
| `scorer` | Named scorer for held-out grading (`exact_match` is wired). | `exact_match` |
| `checkpoint_id` | Label recorded on evaluation records. | `latest` |
| `predictions_filename` | Gradable predictions file written to the output directory. | `predictions.jsonl` |
| `render_predictions` | Whether the final readout prints recursion trees. | `True` |
| `max_render_text_chars` | Truncation length for rendered questions/answers. | `200` |

In [ ]:
# Run-only knobs that live outside the canonical RunSpec.
experiment_config = {
    "scorer": "exact_match",
    "checkpoint_id": "latest",
    "predictions_filename": "predictions.jsonl",
    "render_predictions": True,
    "max_render_text_chars": 200,
}


### Validate and persist

Validate both dicts into their frozen objects (misconfiguration fails here, not mid-run), write them to canonical JSON on Drive, and read them back — the same JSON the `rlm_train` CLI consumes.

In [ ]:
# Validate before launch so misconfiguration fails here, not mid-run.
run_spec = RunSpec.model_validate(run_config)
experiment_settings = ExperimentSettings.model_validate(experiment_config)

RUN_SPEC_PATH = CONFIG_DIR / f"{RUN_NAME}.run_spec.json"
EXPERIMENT_PATH = CONFIG_DIR / f"{RUN_NAME}.experiment.json"
run_spec.write_resolved(RUN_SPEC_PATH)
experiment_settings.write_json(EXPERIMENT_PATH)

# Round-trip back into objects to prove the CLI path reads the same JSON.
run_spec = RunSpec.from_file(RUN_SPEC_PATH)
experiment_settings = ExperimentSettings.from_file(EXPERIMENT_PATH)
OUTPUT_DIRECTORY = Path(run_spec.artifacts.output_directory)

print("Run spec:", RUN_SPEC_PATH)
print("Experiment settings:", EXPERIMENT_PATH)
print("Output directory:", OUTPUT_DIRECTORY)
print("Precision:", run_spec.runtime.precision)
print("Learning rate:", run_spec.runtime.learning_rate)
print("SDPO top_k:", run_spec.objectives.sdpo.top_k)
print("Judge:", run_spec.judge.provider, run_spec.judge.model, run_spec.judge.mode.value)
print("Scorer:", experiment_settings.scorer)
print("Train rows:", sum(1 for _ in Path(TRAIN_PATH).open()))
print("Evaluation rows:", sum(1 for _ in Path(EVAL_PATH).open()))


## Launch training in the background

Runs the canonical CLI (`rlm_train.cli.train`) against the JSON written above. The CLI loads the student model, builds the RLM rollout engine, judge, SDPO objective, and teacher targets from the spec, then optimizes.

In [ ]:
import subprocess
import sys
from pathlib import Path

LOG_PATH = Path("/content/rlm-train-live.log")

log_stream = LOG_PATH.open("w")
training_process = subprocess.Popen(
    [sys.executable, "-u", "-m", "rlm_train.cli.train", str(RUN_SPEC_PATH), "-v"],
    cwd="/content/rlm-ib",
    env=os.environ.copy(),
    stdout=log_stream,
    stderr=subprocess.STDOUT,
)
log_stream.close()

print("Training launched.")
print("PID:", training_process.pid)
print("Run spec:", RUN_SPEC_PATH)
print("Output directory:", OUTPUT_DIRECTORY)
print("Live log:", LOG_PATH)


## Monitor training

Run this cell repeatedly. The first decisive success signal is **Optimizer steps completed: 1**.

In [ ]:
import json
import subprocess

exit_code = training_process.poll()
if exit_code is None:
    print("PROCESS: RUNNING — PID", training_process.pid)
elif exit_code == 0:
    print("PROCESS: COMPLETED SUCCESSFULLY")
else:
    print("PROCESS: FAILED — exit code", exit_code)

print("\nGPU:")
subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.used,memory.total,utilization.gpu",
        "--format=csv,noheader",
    ],
    check=False,
)

metrics_path = OUTPUT_DIRECTORY / "metrics.jsonl"
predictions_path = OUTPUT_DIRECTORY / experiment_settings.predictions_filename

if metrics_path.exists():
    metrics = metrics_path.read_text().splitlines()
    print("\nOptimizer steps completed:", len(metrics))
    if metrics:
        print("Latest metrics:", json.dumps(json.loads(metrics[-1]), indent=2))
else:
    print("\nOptimizer steps completed: 0")

print("Predictions written:", predictions_path.exists())

if LOG_PATH.exists():
    print("\nLATEST LOG:")
    print("\n".join(LOG_PATH.read_text(errors="replace").splitlines()[-80:]))


## Evaluate the held-out set

Runs the recursive policy over the held-out dataset and writes gradable `predictions.jsonl` to the run's output directory.

In [ ]:
import subprocess
import sys

evaluation = subprocess.run(
    [sys.executable, "-u", "-m", "rlm_train.cli.evaluate", str(RUN_SPEC_PATH)],
    cwd="/content/rlm-ib",
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)
print(evaluation.stdout[-4000:])
print(evaluation.stderr[-4000:])
predictions_path = OUTPUT_DIRECTORY / experiment_settings.predictions_filename
print("Predictions written:", predictions_path.exists(), "->", predictions_path)


## Final readout

Load the gradable held-out predictions and print each response with its recursion tree, controlled by the `ExperimentSettings` above.

In [ ]:
import json
from pathlib import Path

from rlm_train.trajectory.render import render_recursion_tree
from rlm_train.trajectory.replay import load_annotated_rollout

predictions_path = OUTPUT_DIRECTORY / experiment_settings.predictions_filename
rollouts_dir = OUTPUT_DIRECTORY / "rollouts"
MAX_SHOWN = 5

if not predictions_path.exists():
    print("No predictions yet at", predictions_path)
else:
    rows = [json.loads(line) for line in predictions_path.read_text().splitlines()]
    print(f"Graded {len(rows)} held-out responses\n")
    for row in rows[:MAX_SHOWN]:
        print(f"[{row['record_id']}] score={row.get('score')}")
        print("Q:", row.get("question"))
        print("A:", row["final_answer"])
        rollout_file = rollouts_dir / f"{row['rollout_id'].replace('/', '_')}.json"
        if experiment_settings.render_predictions and rollout_file.exists():
            rollout = load_annotated_rollout(rollout_file)
            print(render_recursion_tree(rollout, max_text_chars=experiment_settings.max_render_text_chars))
        print("-" * 80)
